# Notebook 2: Machine Architecture & Data Representation
### COMP 1150 — Computer Science Concepts
**Rochester Community and Technical College**

📺 **Lecture video:** *(link coming soon)*

## Learning Outcomes

By the end of this notebook, you will be able to:

- **Explain** how a transistor becomes a logic gate, and how logic gates combine to do arithmetic
- **Describe** the von Neumann architecture and the fetch–decode–execute cycle
- **Convert** numbers between binary, decimal, and hexadecimal by hand
- **Represent** negative numbers using two's complement, and decode a two's-complement byte
- **Explain** how text, images, and sound are all stored as numbers
- **Identify** how a representation choice (like a fixed-size integer) can cause real bugs

*Maps to course LOs: 1 (machine architecture, data representation, processor–memory interaction) and 2 (data storage, number systems, binary)*

## A Memory Budget Crisis

Welcome to **Eight Bits & Bob**, a small game studio shipping its next title onto the **PixelBox 8** — an 8-bit console that, per the box, comes "with up to four colours, two of which are brown."

The game is *Quest for the Reasonably Priced Sword*. The studio director, **Vesper Crunch** — who measures everything in bytes, including lunch — has just delivered the bad news in a meeting: the entire game must fit in a memory budget smaller than a single modern photo. Every character, every sound, every number on screen has to be squeezed into 0s and 1s, efficiently.

That constraint is a gift. It forces us to answer a question most programmers never have to think about: **when everything is just 0s and 1s, how do you store *anything*?** A score? A minus sign? A dragon? A trumpet?

This notebook answers that, from the transistor up.

## The Roadmap

We'll build understanding bottom-up — the same direction a computer is physically built:

| Section | Question it answers |
|---|---|
| A. Transistors → gates → arithmetic | How does a chip *do* anything? |
| B. The von Neumann machine | How are the parts wired together? |
| C. Binary & hexadecimal | How do we write numbers with only 0s and 1s? |
| D. Two's complement | How do we store *negative* numbers? |
| E. Text, images, sound | How is everything else stored as numbers? |
| F. When representation breaks | Why does this matter for real bugs? |

Each section ends with a hands-on activity. Two of them are **practice games built into this notebook** — run them as many times as you want before a quiz.

Notebook 1 closed with a promise: *we'd see how a transistor becomes a logic gate, a logic gate becomes an adder, and an adder becomes a computer.* That's exactly Section A. Let's pay it off.

---
## Section A — From Transistor to Arithmetic

**Dot Mainframe**, the studio's hardware engineer — who named her cat "Cache" — starts at the bottom.

A **transistor** is a tiny electrical switch with no moving parts. It has two states: current flows (call it **1**) or it doesn't (call it **0**). That's it. That's the whole physical foundation of computing. A modern chip has billions of these switches; the PixelBox 8 has a few thousand.

One switch isn't interesting. The magic starts when you wire a few together so their combined behavior follows a *rule*. A small bundle of transistors wired to follow one logical rule is called a **logic gate**.

In [ ]:
# The three gates you must know. Each takes inputs (0 or 1) and produces one output.
from graphviz import Digraph

g = Digraph(graph_attr={"rankdir": "LR"})
g.attr("node", shape="box", style="rounded,filled", fillcolor="lightyellow")

for name in ["AND", "OR", "XOR"]:
    g.node(f"{name}_A", "A", shape="circle", fillcolor="lightgreen")
    g.node(f"{name}_B", "B", shape="circle", fillcolor="lightgreen")
    g.node(f"{name}_G", f"{name}\ngate")
    g.node(f"{name}_Y", "output", shape="circle", fillcolor="lightcoral")
    g.edge(f"{name}_A", f"{name}_G")
    g.edge(f"{name}_B", f"{name}_G")
    g.edge(f"{name}_G", f"{name}_Y")

g.node("NOT_A", "A", shape="circle", fillcolor="lightgreen")
g.node("NOT_G", "NOT\ngate")
g.node("NOT_Y", "output", shape="circle", fillcolor="lightcoral")
g.edge("NOT_A", "NOT_G")
g.edge("NOT_G", "NOT_Y")

g

### How Each Gate Behaves

A gate's rule is captured in a **truth table** — every possible input, and the output it produces. There are no surprises and no choices: same inputs, same output, every time.

**AND** — output 1 only if *both* inputs are 1:

| A | B | A AND B |
|---|---|---|
| 0 | 0 | 0 |
| 0 | 1 | 0 |
| 1 | 0 | 0 |
| 1 | 1 | **1** |

**OR** — output 1 if *either* input is 1. &nbsp;&nbsp; **XOR** ("exclusive or") — output 1 if the inputs are *different*. &nbsp;&nbsp; **NOT** — one input, flipped (0→1, 1→0).

| A | B | A OR B | A XOR B |&nbsp;&nbsp;|&nbsp;A&nbsp;| NOT A |
|---|---|---|---|---|---|---|
| 0 | 0 | 0 | 0 |&nbsp;&nbsp;| 0 | 1 |
| 0 | 1 | 1 | 1 |&nbsp;&nbsp;| 1 | 0 |
| 1 | 0 | 1 | 1 |&nbsp;&nbsp;|&nbsp;|&nbsp;|
| 1 | 1 | 1 | 0 |&nbsp;&nbsp;|&nbsp;|&nbsp;|

Memorize these four. Every calculation a computer has ever done is built from them.

### From Gates to Addition

**Junior Dev Tobble** — who has never seen a number bigger than 255 and would like to keep it that way — asks the obvious question: *how does a pile of gates actually add 1 + 1?*

Look at the XOR and AND tables for the inputs `1` and `1`:

- `1 XOR 1 = 0` → that's the **sum digit** (1 + 1 in binary is `10`, so the digit you write is 0)
- `1 AND 1 = 1` → that's the **carry digit** (the 1 you carry to the next column)

So an XOR gate and an AND gate, side by side, *add two bits*: XOR gives the result digit, AND gives the carry. This little pairing is called a **half-adder**. You don't need to build one — just hold onto the idea: **arithmetic is just gates wired cleverly.**

In [ ]:
# Conceptual view of a half-adder: the same two inputs feed two gates.
ha = Digraph(graph_attr={"rankdir": "LR"})
ha.attr("node", shape="box", style="rounded,filled", fillcolor="lightyellow")

ha.node("A", "bit A", shape="circle", fillcolor="lightgreen")
ha.node("B", "bit B", shape="circle", fillcolor="lightgreen")
ha.node("X", "XOR")
ha.node("N", "AND")
ha.node("S", "SUM digit",   shape="circle", fillcolor="lightcoral")
ha.node("C", "CARRY digit", shape="circle", fillcolor="lightcoral")

ha.edge("A", "X"); ha.edge("B", "X"); ha.edge("X", "S")
ha.edge("A", "N"); ha.edge("B", "N"); ha.edge("N", "C")

ha

Chain eight of these adders together and you can add two 8-bit numbers. That chain is the core of the **ALU** (Arithmetic Logic Unit) — the part of the chip that does math. Add some gates that choose *which* operation to run, and you have, essentially, a processor.

That's the whole stack: **transistor → gate → adder → ALU → CPU.** Promise paid.

### ✏️ Exercise 1 — Gate Arcade &nbsp; 🎮 *Practice game + extend it*

The cell below defines a practice game. Run the cell once to load it, then **follow the instructions in the comment** to play. It generates random gate questions forever — grind it before the quiz.

**Your coding task:** the game only knows AND, OR, and XOR. **Extend it** to also quiz NOT (a one-input gate) *or* add a two-gate combo question (e.g., `(A AND B) OR C`). Then play your improved version and paste your final score in a markdown cell.

In [ ]:
import random

def gate_arcade(rounds=5):
    """Random logic-gate practice. Type 0 or 1 as your answer."""
    gates = {
        "AND": lambda a, b: a & b,
        "OR":  lambda a, b: a | b,
        "XOR": lambda a, b: a ^ b,
    }
    score = 0
    for r in range(1, rounds + 1):
        name = random.choice(list(gates))
        a, b = random.randint(0, 1), random.randint(0, 1)
        correct = gates[name](a, b)
        guess = input(f"Round {r}: {a} {name} {b} = ? ").strip()
        if guess == str(correct):
            print("✅ Correct!")
            score += 1
        else:
            print(f"❌ Not quite — {a} {name} {b} = {correct}")
    print(f"\nFinal score: {score}/{rounds}")

# ▶ TO PLAY: delete the # on the next line, then run this cell.
# gate_arcade()

---
## Section B — The von Neumann Machine

We have a chip that can do math. But where do the numbers *live*, and how do instructions reach the chip?

**Cmdr. Marlow Stack**, the lead engineer — who refers to RAM as "the good china" — explains the layout almost every computer since 1945 has used, the **von Neumann architecture**. Think of the studio's break-room kitchen:

- **CPU** = the cook. Does the actual work, very fast, but holds only a few things at once.
- **Memory (RAM)** = the counter space. Holds what you're working on *right now*. Fast, but cleared when the power goes off.
- **Storage** (the game cartridge) = the pantry. Big, slow, keeps its contents when unplugged.
- **Input/Output** = the serving window. Controller in, picture and sound out.

The crucial von Neumann idea: **the program and its data live in the same memory.** Instructions are just numbers too.

In [ ]:
vn = Digraph(graph_attr={"rankdir": "LR"})
vn.attr("node", shape="box", style="rounded,filled", fillcolor="lightyellow")

vn.node("IN",  "Input\n(controller)",  fillcolor="lightgreen")
vn.node("CPU", "CPU\n(the cook)",      fillcolor="orange")
vn.node("MEM", "Memory / RAM\n(the counter)")
vn.node("OUT", "Output\n(screen + sound)", fillcolor="lightcoral")

vn.edge("IN", "CPU")
vn.edge("CPU", "MEM", label=" read / write ", dir="both")
vn.edge("CPU", "OUT")
vn

### Fast, Small, Expensive vs. Slow, Big, Cheap

Vesper Crunch cares about this because it's literally a budget. Memory comes in a hierarchy — the closer to the CPU, the faster and the more expensive per byte:

| Level | Analogy | Speed | Size | Keeps data without power? |
|---|---|---|---|---|
| Registers | The cook's two hands | Fastest | A handful of numbers | No |
| RAM | The counter | Fast | Limited | No |
| Storage (cartridge/disk) | The pantry | Slow | Large | **Yes** |

This is why a game "loads": it copies data from the slow pantry into the fast counter so the cook can reach it. The whole game can't fit in registers or RAM at once — so the program constantly shuffles data between levels.

In [ ]:
# What the CPU does, over and over, billions of times a second:
fde = Digraph(graph_attr={"rankdir": "TB"})
fde.attr("node", shape="box", style="rounded,filled", fillcolor="lightyellow")

fde.node("F", "FETCH\nget the next instruction from memory", fillcolor="lightgreen")
fde.node("D", "DECODE\nfigure out what it means")
fde.node("E", "EXECUTE\ndo it (math, move data, jump)")
fde.node("S", "store result / advance to next", fillcolor="lightcoral")

fde.edge("F", "D"); fde.edge("D", "E"); fde.edge("E", "S")
fde.edge("S", "F", label=" repeat ")
fde

That loop is the **fetch–decode–execute cycle**. In pseudocode it is almost insultingly simple:

```text
REPEAT FOREVER:
    1. FETCH   the instruction at the current position in memory
    2. DECODE  what that instruction means
    3. EXECUTE it (do math, move data, or jump elsewhere)
    4. ADVANCE to the next instruction
```

Your phone runs this cycle several **billion** times per second. The PixelBox 8 manages a little under two million. The *idea* is identical; only the speed changed — exactly the pattern from Notebook 1.

---
## Section C — Counting in Binary and Hexadecimal

The CPU only has transistors, and a transistor only knows two states. So every number must be written using only **two digits: 0 and 1**. This is **binary** (base 2).

You already know base 10. In `375`, each position is a power of ten: 3×100 + 7×10 + 5×1. Binary works the same way, but each position is a power of **two**.

A group of 8 bits is a **byte**. An 8-bit number has these place values:

In [ ]:
import matplotlib.pyplot as plt

places = [128, 64, 32, 16, 8, 4, 2, 1]
byte   = [0, 1, 0, 0, 1, 0, 1, 1]            # the binary number 01001011
values = [bit * place for bit, place in zip(byte, places)]

plt.figure(figsize=(8, 4))
bars = plt.bar([str(p) for p in places], values, color="teal")
plt.bar([str(p) for p in places], places, color="lightgray", zorder=0)
plt.title("Binary 01001011  →  add the 'on' columns")
plt.xlabel("place value")
plt.ylabel("contribution")
for bar, bit in zip(bars, byte):
    plt.text(bar.get_x() + bar.get_width()/2, 4, str(bit), ha="center")
plt.show()

print("Sum of the 'on' columns:", sum(values))

### Worked Example: `01001011` → decimal

Walk left to right. Wherever the bit is 1, add that column's place value:

| Bit | 0 | 1 | 0 | 0 | 1 | 0 | 1 | 1 |
|---|---|---|---|---|---|---|---|---|
| Place | 128 | 64 | 32 | 16 | 8 | 4 | 2 | 1 |
| Add? | — | 64 | — | — | 8 | — | 2 | 1 |

**64 + 8 + 2 + 1 = 75.** That's the whole method. Going the other way (decimal → binary): subtract the largest place value you can, repeat.

An 8-bit byte holds 0 through 255 (that's 2⁸ = 256 different values). Remember that 255 — Tobble certainly does.

In [ ]:
# Python can convert for you — useful for CHECKING hand work, not replacing it.

n = 75
print(f"{n} in binary: {n:08b}")        # :08b = binary, padded to 8 digits
print(f"{n} in hex:    {n:02X}")        # :02X = hex, 2 digits, uppercase

from_binary = int("01001011", 2)         # the 2 means 'read this as base 2'
print("int('01001011', 2) =", from_binary)

### Hexadecimal: Binary for Humans

**Pip Renderwick**, the studio's sprite and audio artist — who can draw a dragon in 8×8 pixels and insists it's *clearly* a dragon — never writes binary by hand. She uses **hexadecimal** (base 16).

Long binary strings are error-prone for humans (`01001011` — did you miscount?). Hex fixes this because **exactly 4 bits = 1 hex digit**. Hex needs 16 digit symbols, so after 9 it borrows letters:

| Dec | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 | 12 | 13 | 14 | 15 |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| Hex | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | A | B | C | D | E | F |

So `01001011` splits into `0100` (=4) and `1011` (=B) → **`4B`**. Two readable characters instead of eight bits. This is why colors look like `#4B2C9F` and memory addresses look like `0x1A3F`.

In [ ]:
import numpy as np

fig, ax = plt.subplots(figsize=(10, 2.2))
for d in range(16):
    ax.text(d, 1.0, f"{d:04b}", ha="center", family="monospace")
    ax.text(d, 0.6, f"{d:X}",   ha="center", family="monospace", color="crimson", fontsize=14)
    ax.text(d, 0.2, str(d),     ha="center", family="monospace")
ax.text(-1.4, 1.0, "binary", ha="left"); ax.text(-1.4, 0.6, "hex", ha="left")
ax.text(-1.4, 0.2, "decimal", ha="left")
ax.set_xlim(-1.5, 16); ax.set_ylim(0, 1.3); ax.axis("off")
ax.set_title("One reference strip: 4 bits ↔ 1 hex digit")
plt.show()

Hex isn't a *different* number system from binary in any deep sense — it's a compact, human-friendly way to write the same bits. You'll see it constantly: colors, memory addresses, error codes, file signatures. Practice it in the arcade after the next section.

---
## Section D — Representing Negative Numbers

Tobble hits a wall. *Quest for the Reasonably Priced Sword* needs to track when the hero falls in a pit: **−50 points**. But a byte is just eight 0s and 1s. **Where does the minus sign go?**

**Naive idea — "sign bit":** use the leftmost bit as a sign (0 = positive, 1 = negative), and the other 7 bits as the size. This is called *sign-magnitude*, and it almost works. But it has two ugly problems:

- There are **two zeros**: `00000000` (+0) and `10000000` (−0). That's wasteful and breaks comparisons.
- Addition stops working normally — the hardware needs special cases for the sign.

Computers use a cleverer scheme that has only one zero and lets the *same* adder handle positive and negative numbers: **two's complement**.

### The Two's Complement Rule

In an 8-bit two's-complement system:

- The leftmost bit still signals sign: **0 = non-negative, 1 = negative**.
- **Positive numbers** look exactly like normal binary (`00000101` = 5).
- To get **−x**: take the binary for `x`, **flip every bit**, then **add 1**.

That's the whole rule: **invert, then add one.** The range an 8-bit byte can hold shifts to **−128 through +127** (still 256 distinct values, still one zero).

Why does "invert and add 1" work? Because it makes `x + (−x)` overflow cleanly to zero in 8 bits — so the ordinary adder from Section A just works, no special cases. The hardware never even knows the number was "negative."

### Worked Example 1: encode **−37** as an 8-bit byte

| Step | Result |
|---|---|
| 1. Write +37 in binary | `00100101` |
| 2. Flip every bit | `11011010` |
| 3. Add 1 | `11011011` |

So **−37 = `11011011`**. Notice the leftmost bit is 1 — correctly signaling "negative."

**Quick check:** add `00100101` (+37) and `11011011` (−37). Add column by column, discard any 9th-bit carry → `00000000`. It sums to zero, exactly as a number and its negative should.

### Worked Example 2: decode `11011011` back to a signed number

The leftmost bit is **1**, so this byte is negative. To find its value, reverse the process (invert, add 1) to get the magnitude:

| Step | Result |
|---|---|
| Given byte (leftmost bit 1 → negative) | `11011011` |
| 1. Flip every bit | `00100100` |
| 2. Add 1 | `00100101` |
| 3. That's the magnitude in decimal | 37 |
| 4. Apply the negative sign | **−37** |

**Exam tip:** if the leftmost bit is 0, just read it as normal binary. If it's 1, do *flip-and-add-1* to find the magnitude, then put a minus sign on it. That single rule answers every two's-complement quiz question.

In [ ]:
# A 4-bit 'number wheel' shows why negatives wrap around from the top.
import numpy as np

fig, ax = plt.subplots(figsize=(5.5, 5.5))
for raw in range(16):
    angle = np.pi/2 - raw * (2*np.pi/16)
    x, y = np.cos(angle), np.sin(angle)
    signed = raw if raw < 8 else raw - 16          # 4-bit two's complement
    ax.text(x*1.18, y*1.18, f"{raw:04b}", ha="center", va="center",
            family="monospace", fontsize=9)
    color = "navy" if signed >= 0 else "crimson"
    ax.text(x*0.85, y*0.85, str(signed), ha="center", va="center",
            color=color, fontsize=13, fontweight="bold")
ax.add_patch(plt.Circle((0, 0), 1.0, fill=False, color="gray"))
ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5)
ax.set_aspect("equal"); ax.axis("off")
ax.set_title("4-bit two's complement: bits (outer) vs. signed value (inner)")
plt.show()

Notice on the wheel: as the bits climb past the halfway point (`1000`), the *meaning* drops to the most negative value and climbs back toward −1. The numbers wrap around like an odometer. Python's own integers don't have this limit (they grow as large as needed), but **fixed-width** values — registers, image pixels, the PixelBox 8's score counter — absolutely do. We can simulate a fixed 8-bit value with a *mask*:

In [ ]:
value = -37
byte  = value & 0xFF          # & 0xFF keeps only the lowest 8 bits
print(f"{value} stored in 8 bits: {byte:08b}  (raw value {byte})")

# Reading it back as signed:
raw = 0b11011011
signed = raw - 256 if raw >= 128 else raw   # leftmost bit set → subtract 256
print("11011011 interpreted as signed:", signed)

### ✏️ Exercise 2 — Number Base Arcade &nbsp; 🎮 *Practice game + tweak it*

The next cell defines the notebook's main test-prep tool. It has four modes:

| Mode | Asks you to… |
|---|---|
| `"bin2dec"` | read a binary byte as decimal |
| `"dec2bin"` | write a decimal number as an 8-bit byte |
| `"hex"` | give the 2-digit hex for a decimal number |
| `"twos"` | give the 8-bit two's-complement byte for a signed number |

**Self-study:** play every mode until you can hit a streak of 5. **Coding task:** make one improvement — add a `"hex2bin"` mode, a timer, or extend `"bin2dec"` to 16-bit. Paste your scores and describe your change in a markdown cell.

In [ ]:
import random

def base_arcade(mode="bin2dec", rounds=5):
    """Random number-base practice. Modes: bin2dec, dec2bin, hex, twos."""
    score = 0
    for r in range(1, rounds + 1):
        if mode == "bin2dec":
            n = random.randint(0, 255)
            prompt, correct = f"Round {r}: binary {n:08b} = ? (decimal) ", str(n)
        elif mode == "dec2bin":
            n = random.randint(0, 255)
            prompt, correct = f"Round {r}: decimal {n} = ? (8-bit binary) ", f"{n:08b}"
        elif mode == "hex":
            n = random.randint(0, 255)
            prompt, correct = f"Round {r}: decimal {n} = ? (2-digit hex) ", f"{n:02X}"
        elif mode == "twos":
            n = random.randint(-128, 127)
            prompt, correct = f"Round {r}: {n} as an 8-bit two's-complement byte = ? ", f"{n & 0xFF:08b}"
        else:
            print("Unknown mode. Use bin2dec, dec2bin, hex, or twos.")
            return
        guess = input(prompt).strip().upper()
        if guess == correct:
            print("✅ Correct!")
            score += 1
        else:
            print(f"❌ Answer was {correct}")
    print(f"\n[{mode}] score: {score}/{rounds}")

# ▶ TO PLAY: delete the # on ONE line below, then run this cell.
# base_arcade("bin2dec")
# base_arcade("dec2bin")
# base_arcade("hex")
# base_arcade("twos")

---
## Section E — Encoding Text, Images, and Sound

Pip Renderwick's rule for the whole studio: **"if it's in the game, it's a number somewhere."** Numbers were the easy part — they *are* numbers. Now the trick is turning everything else into numbers too.

### Text = Numbers in a Lookup Table

A computer stores the letter `A` by agreeing, in advance, that `A` *is* the number 65. That agreement is a **character encoding**. The classic one is **ASCII**: `A`=65, `B`=66, `a`=97, `space`=32, and so on.

You already met this in Notebook 1's Caesar cipher — `ord()` gives a character's code, `chr()` turns a code back into a character. ASCII only covers 128 characters (enough for English). Modern systems use **Unicode**, which extends the same idea to every writing system on Earth plus emoji — `🐉` is just a (large) number with an agreed meaning.

In [ ]:
import numpy as np

codes = np.arange(32, 128).reshape(8, 12)     # printable ASCII
fig, ax = plt.subplots(figsize=(10, 5))
ax.imshow(codes, cmap="viridis")
for i in range(8):
    for j in range(12):
        c = int(codes[i, j])
        ax.text(j, i, f"{chr(c)}\n{c}", ha="center", va="center",
                color="white", fontsize=8)
ax.set_title("Printable ASCII: every character is just a number")
ax.axis("off")
plt.show()

### Images = Grids of Numbers

A digital image is a grid of **pixels**, and each pixel is a number. The simplest case: 1 bit per pixel — 0 is background, 1 is drawn. That's exactly how a PixelBox 8 sprite works. Color just means more bits per pixel (and a small **palette** — Vesper only paid for four colors). The image below is rendered straight from a grid of 0s and 1s.

In [ ]:
# Pip's 8x8 'definitely a dragon' sprite — a grid of bits.
dragon = [
    [0, 0, 1, 1, 0, 0, 0, 0],
    [0, 1, 1, 1, 1, 0, 0, 0],
    [1, 1, 0, 1, 1, 1, 0, 0],
    [1, 1, 1, 1, 1, 1, 1, 0],
    [0, 1, 1, 1, 1, 1, 1, 1],
    [0, 0, 1, 1, 1, 1, 1, 0],
    [0, 1, 1, 0, 0, 1, 1, 0],
    [1, 1, 0, 0, 0, 0, 1, 1],
]

plt.figure(figsize=(3, 3))
plt.imshow(dragon, cmap="binary")
plt.title("8 bytes = one sprite")
plt.axis("off")
plt.show()

### Sound = Numbers Over Time

Sound is a wave. To store it, the computer measures the wave's height many times per second; each measurement is a number (**sampling**). The PixelBox 8's audio chip is cheap, so instead of smooth waves it mostly produces blocky **square waves** — the classic "chiptune" buzz. Same idea, just a crude shape:

In [ ]:
import numpy as np

t = np.linspace(0, 1, 600)
sine   = np.sin(2 * np.pi * 4 * t)
square = np.sign(sine)

plt.figure(figsize=(9, 3))
plt.plot(t, sine,   label="smooth wave (real world)")
plt.plot(t, square, label="square wave (PixelBox 8 audio)")
plt.legend(loc="upper right")
plt.title("A sound, stored as a list of numbers over time")
plt.yticks([-1, 0, 1])
plt.show()

Text, image, sound: three completely different experiences, **one underlying idea** — a sequence of numbers plus an agreed rule for what those numbers mean. Change the rule and the same bytes become gibberish. That's the bridge to the final section.

### ✏️ Exercise 3 — Design Your Own Sprite &nbsp; 🎨 *Edit the code directly*

Edit the `0`s and `1`s in the cell below to draw your own 8×8 sprite for *Quest for the Reasonably Priced Sword* (a sword? a coin? a slime?). Run the cell to see it.

**Optional bonus — animation:** fill in the second grid `frame2` with a slightly different pose, then run the bottom block to flip between them. That's a two-frame animation — exactly how 1980s games made things "move" on a tiny memory budget.

In [ ]:
# ✏️ EDIT THESE 0s and 1s — this cell is yours to change.
frame1 = [
    [0, 0, 0, 1, 1, 0, 0, 0],
    [0, 0, 0, 1, 1, 0, 0, 0],
    [0, 0, 0, 1, 1, 0, 0, 0],
    [0, 0, 0, 1, 1, 0, 0, 0],
    [0, 0, 1, 1, 1, 1, 0, 0],
    [0, 1, 1, 1, 1, 1, 1, 0],
    [0, 0, 0, 1, 1, 0, 0, 0],
    [0, 0, 0, 1, 1, 0, 0, 0],
]

# Optional second frame for the animation bonus (start it as a copy, then tweak):
frame2 = [
    [0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 1, 1, 0, 0, 0],
    [0, 0, 0, 1, 1, 0, 0, 0],
    [0, 0, 0, 1, 1, 0, 0, 0],
    [0, 0, 1, 1, 1, 1, 0, 0],
    [0, 1, 1, 1, 1, 1, 1, 0],
    [0, 0, 0, 1, 1, 0, 0, 0],
    [0, 0, 0, 1, 1, 0, 0, 0],
]

plt.figure(figsize=(3, 3))
plt.imshow(frame1, cmap="binary")
plt.title("Your sprite")
plt.axis("off")
plt.show()

In [ ]:
# ▶ ANIMATION BONUS: run this after filling in frame2.
import time
from IPython.display import clear_output

for _ in range(6):                       # 6 flips, then stop
    for frame, label in [(frame1, "frame 1"), (frame2, "frame 2")]:
        clear_output(wait=True)
        plt.figure(figsize=(3, 3))
        plt.imshow(frame, cmap="binary")
        plt.title(label)
        plt.axis("off")
        plt.show()
        time.sleep(0.3)

---
## Section F — When Representation Breaks

**Quoth Bugbear**, the studio's QA tester — who finds the bug, was right all along, and is insufferable about it — files report #256:

> *"At level 256, the game corrupts. Enemies turn into garbage tiles. Half the screen is hex soup. Unplayable."*

The level counter is stored in **one byte**. A byte holds 0–255. When the game adds 1 to level 255, the value doesn't go to 256 — there's no room for a 9th bit, so it **wraps around to 0** (or worse, into memory it shouldn't touch). This is **integer overflow**, and it's not a fictional problem.

### The Real One: the Pac-Man Kill Screen

The 1980 arcade game **Pac-Man** stored its level number in a single byte and used it to draw the bonus-fruit display. At **level 256**, that byte overflowed. The level-draw routine ran wild and corrupted the right half of the screen into a famous mess of garbled tiles — the "**kill screen**." The game was literally unbeatable past that point, not by design, but because of one undersized integer.

| Decision | Consequence |
|---|---|
| Store level in 1 byte | Saves memory (precious in 1980) |
| Never expect 256 levels | Reasonable — almost no one got there |
| No overflow check | Level 256 corrupts memory → kill screen |

Same bug class still bites today: a 2015 Boeing 787 advisory warned that a counter could overflow after 248 days of continuous power, potentially shutting down electrical generators mid-flight. **Representation size is a safety decision.**

In [ ]:
# Simulate an 8-bit level counter. & 0xFF forces it to stay 8 bits wide.
level = 253
for _ in range(6):
    level = (level + 1) & 0xFF
    note = "  <-- overflow! wrapped around" if level == 0 else ""
    print(f"next level -> stored as {level:3d}  ({level:08b}){note}")

Run that cell and watch 254 → 255 → **0**. Nothing is "broken" — the hardware did exactly what two's-complement-style wrap-around always does. The bug was a *human* decision: choosing a box too small for the numbers that would eventually go in it.

Every representation choice in this notebook — byte width, palette size, sample rate, encoding — is an engineering trade-off with real consequences. That's the deep lesson of data representation, and it's why this is Section 1 and 2 of the course, not an afterthought.

### ✏️ Exercise 4 — Build a Retro Toy with AI &nbsp; 🤖 *AI-Assisted*

Use Gemini (or Claude / ChatGPT) to build **one small, fun thing** that runs in this notebook and uses a concept from this notebook. Pick one:

| Option | What it does | Concept it uses |
|---|---|---|
| **Secret Message Machine** | Turns text → binary or hex, and back again | `ord`/`chr`, binary/hex |
| **Pixel Banner** | Turns your initials into blocky 8-bit letters with `imshow` | bitmaps / sprite grids |
| **Chiptune Plotter** | Turns a few notes into a plotted square-wave "melody" | sound = numbers, square waves |
| **Overflow Survival** | A tiny text game where your score is an 8-bit byte and wraps at 256 | two's complement / overflow |

**Steps:**
1. Ask the AI to draft your chosen toy. Be specific about it running in a Jupyter/Colab cell.
2. Get it actually working in a new code cell (it probably won't run perfectly on the first try).
3. In a markdown cell, write 2–3 sentences: **what did the AI get wrong or what did you improve, and how did you fix it?**

Remember the course rule: *AI is a fast first draft. You verify.*

---
## Key Terms

- **Transistor** — A microscopic electrical switch with two states (on/off); the physical basis of all digital computing.
- **Logic gate** — A small group of transistors that follows one logical rule (AND, OR, NOT, XOR).
- **Truth table** — A table listing every input combination for a gate or circuit and the resulting output.
- **Half-adder** — An XOR gate plus an AND gate that together add two bits (XOR = sum digit, AND = carry digit).
- **ALU (Arithmetic Logic Unit)** — The part of the CPU that performs arithmetic and logic, built from chained adders and gates.
- **von Neumann architecture** — The standard design where CPU, memory, and I/O are connected and program instructions live in the same memory as data.
- **CPU** — The processor; fetches and executes instructions.
- **Register** — A tiny, extremely fast storage location inside the CPU.
- **RAM** — Fast main memory that holds active data; loses its contents when power is removed.
- **Storage** — Slow, large, persistent memory (disk, cartridge) that keeps data without power.
- **Fetch–decode–execute cycle** — The repeating loop a CPU performs to run a program.
- **Bit** — A single binary digit, 0 or 1.
- **Byte** — 8 bits; holds one of 256 values (0–255 unsigned, −128–127 signed).
- **Binary (base 2)** — A number system using only 0 and 1; place values are powers of two.
- **Hexadecimal (base 16)** — A compact, human-friendly notation where 4 bits map to one hex digit (0–9, A–F).
- **Two's complement** — The standard scheme for storing signed integers: negate by flipping all bits and adding 1.
- **Integer overflow** — When a value exceeds the maximum its fixed-size storage can hold and wraps around.
- **Character encoding** — An agreed mapping from characters to numbers (ASCII, Unicode).
- **ASCII / Unicode** — Standard character encodings; ASCII covers 128 characters, Unicode covers virtually all writing systems.
- **Pixel** — One dot of an image, stored as a number; more bits per pixel means more possible colors.
- **Sampling** — Measuring a sound wave's height many times per second to store it as a list of numbers.
- **Palette** — A small fixed set of colors an image is allowed to use, to save memory.

---
## Summary

- **Transistors → gates → adders → CPU.** Computation is switches wired to follow logical rules.
- **von Neumann architecture** connects CPU, memory, and I/O; the CPU runs a fetch–decode–execute loop billions of times per second.
- **Binary** is counting with powers of two; **hexadecimal** is the same bits written compactly for humans.
- **Two's complement** stores negative numbers so the ordinary adder still works — *flip the bits, add one*.
- **Text, images, and sound** are all numbers plus an agreed rule for interpreting them.
- **Representation size is an engineering and safety decision** — the Pac-Man kill screen was one byte, chosen too small.

Everything is numbers. The size of the box you put them in matters.

---
## What's Next

You now know what the machine *is* and how it stores things. **Notebook 3** turns to instructing it: we'll go from a plain-English description of a problem, to **pseudocode**, to a **flowchart**, to working **Python** — the authoring workflow this whole course runs on. Eight Bits & Bob still has a game to finish, and the code isn't going to write itself. (Bob keeps insisting it will. Nobody listens to Bob.)

---
*COMP 1150 — Computer Science Concepts · Rochester Community and Technical College*  
*Content licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Notebook authored with AI assistance; all content reviewed and edited by the instructor.*